# Expirements for the Image Sticthing App

In [ ]:
import os
import cv2 
import numpy as np
import pandas as pd
import networkx as nx
from itertools import compress
import matplotlib.pyplot as plt
import main
import utils.file_utils as file_utils
import models

## Using OpenCV Sticther

In [ ]:
img_paths = file_utils.get_image_files("/home/schmidtg/coding_projetcs/local-image-features-apps/src/image_stitching/datasets/example-data/flower")
images = [cv2.imread(p) for p in img_paths]

stitcher = cv2.Stitcher_create()
status, panorama = stitcher.stitch(images)

if status == cv2.Stitcher_OK:
    cv2.imwrite("../../output/cv_panorama.jpg", panorama)
    print("Panorama saved!")
else:
    print("Error during stitching:", status)

## Critical Steps
All the features accross all the images need to be detetected, described and matched with each other. 

Important aspects:
1. Detect all features
2. Compute all descriptors
2. Save ids of features
3. Know which features belong to which image
5. Match all features with closest K neighbours
6. Remove invalid matches (features from the same image)

### Detect All Features and Compute Descriptors

In [ ]:
img_paths = file_utils.get_image_files("/home/schmidtg/coding_projetcs/local-image-features-apps/src/image_stitching/datasets/example-data/myself")
orb = cv2.ORB_create()

img_id_bounds = np.array([0], dtype=np.uint32)  # indicated at which ID an image starts/end
N = 0
des = np.empty((0,32), dtype=np.uint8) # data type is crucial!
img_ids = np.empty((0))
kps = ()

for img_id, img_path in enumerate(img_paths): 
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    kp_new, des_new = orb.detectAndCompute(img, None)  # type: ignore
    des = np.vstack((des, des_new))
    kps +=  kp_new
    N_new = len(des_new)
    N += N_new
    img_id_bounds = np.append(img_id_bounds, N)
    img_ids_new = np.full((N_new), img_id)
    img_ids = np.concatenate((img_ids, img_ids_new), axis=0)

In [ ]:
print("Total features detected accross images:\n\t", len(kps))
print("dType of kps:\n\t", type(kps))
print("dType of elements in kps:\n\t", type(kps[0]))
print("Id of the first feature belonging to each image\n\t", img_id_bounds)
print("img ids of each feature:\n\t", img_ids)
print("Descriptors of features:\n", des)

### Save Features with Descriptors and ImageID by Feature ID into Pandas Dataframe

In [ ]:
matching_df = pd.DataFrame(
    {"Descriptor" : list(des),
     "ImgId" : img_ids})
convert_dict ={"ImgId": np.uint8}  # dont expect more than 2^8 / 255 images
matching_df = matching_df.astype(convert_dict)
print(matching_df.dtypes)
print("Descriptor dtype:", type(matching_df["Descriptor"].iloc[0]), matching_df["Descriptor"].iloc[0].dtype)
matching_df.head()

### Macth All Features

In [ ]:
# get descriptors from table into correct format
x = np.stack(list(matching_df["Descriptor"])) # x = des


# Define FLANN parameters and match descriptors
# from https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html
FLANN_INDEX_LSH = 6
index_params= dict(algorithm = FLANN_INDEX_LSH,
                   table_number = 6, # 12
                   key_size = 12,     # 20
                   multi_probe_level = 1) #2
search_params = dict(checks=50)   # or pass empty dictionary
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(x,x, k=5)

# 2nd Method, Not Working
# FLANN_INDEX_KDTREE = 0
# index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
# search_params = dict(checks=50)
# flann = cv2.FlannBasedMatcher(index_params, search_params)
# matches = flann.knnMatch(des, des, k=2)

In [ ]:
# we see that the 1st match, is always matched the result of the keypoint being matched with itself
for M in matches[0:10]:
    m = M[0]
    print(m.trainIdx, m.queryIdx)

In [ ]:
type(matches), type(matches[0]), type(matches[0][0])

In [ ]:
# so remove the 1st match for all of them - redundant because of remove_invalid_matches
matches1 = tuple((m[1:]) for m in matches)

In [ ]:
matches_list, matched_col = list(), list()

for i, m_tuple in enumerate(matches):
    m_list = list(m_tuple[1:])
    matches_list.append(m_list)
    matched_col.append(np.array([m.trainIdx for m in m_list]))

In [ ]:
type(matches_list), type(matches_list[0]), type(matches), type(matches)

In [ ]:
matched_col[0:5]

In [ ]:
matching_df['Matched_Ids'] = matched_col

In [ ]:
matching_df.head()

In [ ]:
# convert matched descriptor ids to img ids 
def descriptor_id_to_img_id(id,bounds) -> int:
    """
    Get the associated Image ID for the given Descriptor ID, using the given bounds. 
    Returns:
        int: in range from 0 to length of bounds-1
    """
    return np.searchsorted(bounds, id, side='right') - 1

def matched_ids_to_img_ids(row, bounds):
    matched_ids = row['Matched_Ids']
    img_ids = [descriptor_id_to_img_id(id, bounds) for id in matched_ids]
    return np.array(img_ids)

matching_df['Matched_ImgIds'] = matching_df.apply(matched_ids_to_img_ids, 
                                                  axis=1, 
                                                  args=(img_id_bounds,))

In [ ]:
for m in matches_list[0:5]:
    print(len(m))

In [ ]:
matching_df.head()

In [ ]:
# remove Matched Ids of descriptors that belong to the same image
def remove_invalid_matches(row, matches_list: list) -> pd.Series:
    """
    Removes Ids from Matched_Ids that are the are from the same Image (ImgId).
    Also modifies the list matches_list passed as an argument.
    Args:
        row: a row from a pandas dataframe

    Returns:
        series: a series for containing the updated Matched_Ids and Matched_ImgIds, with invalid values removed.
    """
    index, matched_ids, img_ids, imgId = int(row.name), row['Matched_Ids'], row['Matched_ImgIds'], row['ImgId'] 
    indeces_to_remove = np.where(img_ids == imgId)
    # create mask to remove values
    mask = np.ones(len(matched_ids), dtype=bool)
    mask[indeces_to_remove] = False
    matched_ids = matched_ids[mask,...]
    img_ids = img_ids[mask,...]
    matches_list[index] = list(compress(matches_list[index], mask))
    return pd.Series([matched_ids, img_ids])

In [ ]:
matching_df[["Matched_Ids", "Matched_ImgIds"]] = matching_df.apply(remove_invalid_matches, axis=1, args=(matches_list,))
matching_df.head()

In [ ]:
for m in matches_list[0:5]:
    print(len(m))

In [ ]:
np.stack(matching_df[(matching_df["ImgId"] == 0)]["Descriptor"])

## Determine How Many Matches Between Images

In [ ]:
# Get the counts of imag matches
img_id_stats_df = matching_df.explode('Matched_ImgIds')
result = (
    img_id_stats_df
    .groupby(['ImgId', 'Matched_ImgIds'])
    .size()
    .reset_index(name='Count')
    .rename(columns={'Matched_ImgIds': 'MatchedWith'})
)

print(result)

In [ ]:
# Pivot the data so each "MatchedWith" becomes a column
pivot_df = result.pivot(index='ImgId', columns='MatchedWith', values='Count').fillna(0)

# Plot grouped bars
pivot_df.plot(kind='bar', figsize=(8,5))

plt.title("Descriptor Match Counts per Image")
plt.xlabel("Image ID")
plt.ylabel("Match Count")
plt.legend(title="Matched With")
plt.tight_layout()
plt.show()

## Match Ratio Between Images 

In [ ]:
n_images = len(img_paths)
match_ratio = np.zeros((n_images,n_images))
i = 0

for row in range (n_images):
    for col in range (n_images):
        if row == col:
            continue
        first_id = img_id_bounds[row]
        last_id = img_id_bounds[row+1]
        n_features = last_id - first_id
        n_matches = result.iloc[i]['Count']
        match_ratio[row][col] = n_matches / n_features
        i+=1

In [ ]:
fig, ax = plt.subplots()

cax = ax.imshow(match_ratio)

# Add colorbar
plt.colorbar(cax)

# Axis labels (optional)
 
labels = [f"{i}" for i in range(n_images)]
ax.set_xticks(range(n_images))
ax.set_yticks(range(n_images))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)

# Rotate x labels for readability
plt.xticks(rotation=45)

# Annotate each cell with value
for i in range(n_images):
    for j in range(n_images):
        display_value = f"{int(match_ratio[i, j]*100)}%"
        ax.text(j, i, display_value, ha="center", va="center")

plt.xlabel("Reference Image")
plt.ylabel("Target Image")
plt.title("Match Percentage")

plt.tight_layout()
plt.show()

## Connectivity Graph From Match Ratios

In [ ]:
fully_connected_graph = nx.Graph()

for i in range(n_images):
    for j in range(i+1, n_images):
        mutual_ratio = match_ratio[i,j] 
        mutual_ratio += match_ratio[j,i]
        mutual_ratio /= 2
        if mutual_ratio > 0.25:
            inverse_ratio = 1 - mutual_ratio
            fully_connected_graph.add_edge(i, j, weight=inverse_ratio)  # invert for MST

pos = nx.spring_layout(fully_connected_graph)

# Draw graph
nx.draw(fully_connected_graph, pos, with_labels=True)

for i, path in enumerate(img_paths):
    print(f"{i}: {os.path.basename(path)}")
plt.show()


In [ ]:
# optimised graph
pano_graph = nx.minimum_spanning_tree(fully_connected_graph)

pos = nx.spring_layout(pano_graph)

# Draw graph
nx.draw(pano_graph, pos, with_labels=True)

for i, path in enumerate(img_paths):
    print(f"{i}: {os.path.basename(path)}")
plt.show()


### Most Well Connected Image

In [ ]:
# Compute closeness centrality for all nodes
closeness = nx.closeness_centrality(pano_graph)

# Find the node with the highest centrality
best_node = max(closeness, key=closeness.get)

print("Best connected node:", best_node)

In [ ]:
top_matches = (
    result
    .sort_values(['ImgId', 'Count'], ascending=[True, False])
    .groupby('ImgId')['MatchedWith']
    .apply(list)
    .to_dict()
)

top_matches

## Determine Matches Between 2 specific Images From matching_df

1. Reduce matching_df to entries containing reference image (ImageID) and has target Image in "Matched_ImgIds" column.
2. Create macth tuple for each row, saying which keypoint from reference is matched to which keypoint in traget.
3. Combine the match tuples from all the rows in the   

In [ ]:
matching_df.head()

In [ ]:
def get_sub_matching_dataframe(id1, id2, matching_df):
    """
    Reduces matching_df to a subset containing features that are matched between 2 specific images. 

    Args:
        id1 (int): what 'ImgId' column should be equal to
        id2 (int): what should be in 'Matched_ImgIds' column
        matching_df (pandas dataframe): 

    Returns:
        pandas dataframe: subset of matching_df
    """
    sub_matching_df = matching_df[
        (matching_df["ImgId"] == id1) &
        (matching_df['Matched_ImgIds'].apply(lambda x: id2 in x))
        ]
    return sub_matching_df

def generate_id_pair(row, id2: int):
    feature_id_img1 = int(row.Index)  # id of feature in reference image 
    matched_ids = np.array(row.Matched_Ids)  # ids of matched feature
    img_ids = np.array(row.Matched_ImgIds)  # ids of images of matched features
    mask = img_ids == id2
    valid_id = matched_ids[mask]
    new_entries = [(feature_id_img1, int(feature_id_img2)) for feature_id_img2 in valid_id]
    return new_entries


def determine_matches_between_image_pair(id1, id2, matching_df):
    sub_matching_df = get_sub_matching_dataframe(id1, id2, matching_df)
    id_matches = [] 
    for row in sub_matching_df.itertuples(index=True):
        id_matches.extend(generate_id_pair(row, id2))
    return id_matches


In [ ]:
id1 = 1
id2 = 0
matched_ids = determine_matches_between_image_pair(id1, id2, matching_df)
matched_ids

In [ ]:
# image to match / stitch
get_sub_matching_dataframe(id1, id2, matching_df)

In [ ]:
def get_homogrphy(kps, matched_ids):
    src = np.empty((len(matched_ids),2), dtype=np.float32)  # coords of features from reference image 
    dst = np.empty((len(matched_ids),2), dtype=np.float32)  # coords of features from target image 

    for i, id_pair in enumerate(matched_ids):
        kp_id1, kp_id2 = id_pair  # get ids of features 
        kp1, kp2 = kps[kp_id1], kps[kp_id2]  # use ids of features to get their objs
        src[i,0] = kp1.pt[0]  # x coord
        src[i,1] = kp1.pt[1]  # y coord
        dst[i,0] = kp2.pt[0]  # x coord
        dst[i,1] = kp2.pt[1]  # y coord

    H, _ =  cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    return H  # transfromion to go from target to reference  

H = get_homogrphy(kps, matched_ids)

In [ ]:
H

In [ ]:
img1 = cv2.imread(img_paths[id1], cv2.IMREAD_COLOR_RGB)
img2 = cv2.imread(img_paths[id2], cv2.IMREAD_COLOR_RGB)

In [ ]:


# def stitch_images(img1, img2, H):
#     h1, w1 = img1.shape[:2]
#     h2, w2 = img2.shape[:2]

#     # Get the canvas dimesions
#     pts = np.float32([[0, 0], [0, h1], [w1, h1], [w1, 0]]).reshape(-1, 1, 2)
#     img2_warped = cv2.warpPerspective(img2, H, (w1 + w2, h1))

#     # Place the first image on the canvas
#     img2_warped[0:h1, 0:w1] = img1
#     return img2_warped

# stitched = stitch_images(img1, img2, H)

# # Display inline
# plt.imshow(stitched, cmap="gray")
# plt.axis('off')  # hide axes
# plt.show()

In [ ]:
def stitch_images2(img1, img2, H):
    """
    Stitch img2 onto img1 using homography H (img2 -> img1)
    img1 is the base canvas
    """
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    # Corners of img2
    corners_img2 = np.array([[0,0],[0,h2],[w2,h2],[w2,0]], dtype=np.float32).reshape(-1,1,2)
    # Warp corners into img1's frame using H
    warped_corners = cv2.perspectiveTransform(corners_img2, H)

    # Corners of img1
    corners_img1 = np.array([[0,0],[0,h1],[w1,h1],[w1,0]], dtype=np.float32).reshape(-1,1,2)

    # All corners together to find canvas size
    all_corners = np.concatenate((warped_corners, corners_img1), axis=0)
    [xmin, ymin] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
    [xmax, ymax] = np.int32(all_corners.max(axis=0).ravel() + 0.5)

    # Translation matrix to shift everything into positive coordinates
    translation = np.array([[1,0,-xmin],[0,1,-ymin],[0,0,1]])

    # Warp img2 into canvas
    result = cv2.warpPerspective(img2, translation @ H, (xmax - xmin, ymax - ymin))
    # Place img1 into canvas
    result[-ymin:h1 - ymin, -xmin:w1 - xmin] = img1

    return result

stitched = stitch_images2(img2, img1, H) # image 2 is the base actually, 

# Display inline
plt.imshow(stitched, cmap="gray")
plt.axis('off')  # hide axes
plt.show()

In [ ]:
type(stitched)

In [ ]:
paths = file_utils.get_image_files("/home/schmidtg/coding_projetcs/local-image-features-apps/src/image-stitching/datasets/example-data/myself")
models.Sticth
